# Pattern 10: Agentic RAG

Follows SPEC.md §8's mandatory 8-section template -- this is one of "the 10 patterns."

**This notebook's committed execution uses `RAG_RECIPES_LLM=mock`** for both the embedding and
generation/agent-decision steps.

Per SPEC.md §6's special note for this pattern, every tool-call trace is logged to
`outputs/agentic_traces.jsonl` -- section 5 below passes `trace_output_path` straight into
`run_pattern()` (see `evals/run.py`), so this happens as a natural side effect of the existing eval
step, not a separate/duplicate retrieval pass. Section 7 is a PENDING placeholder awaiting a
real-embeddings-and-LLM run (see `tasks/todo.md`).


## Reproducibility header (SPEC.md §11)

In [1]:
import platform
import sys
import subprocess
import openai
import numpy

print(f"platform: {platform.platform()}")
print(f"python: {sys.version}")
print(f"openai sdk: {openai.__version__}")
print(f"numpy: {numpy.__version__}")

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_sha = "(not in a git repo checkout)"
print(f"git commit: {git_sha}")


platform: Windows-11-10.0.26200-SP0
python: 3.12.13 (main, Aug  7 2026, 02:26:41) [MSC v.1944 64 bit (AMD64)]
openai sdk: 2.53.0
numpy: 2.5.2
git commit: 28c83500bd3d1640831edd8eb96023292300812b


## Setup (loaded once, used by every section below)

In [2]:
import os
os.environ.setdefault("RAG_RECIPES_LLM", "mock")

from evals.run import load_corpus_by_id, load_qa_set, run_pattern
from recipes.llm import get_llm, MockLLM

corpus_by_id = load_corpus_by_id("../corpus/corpus.jsonl")
qa_set = load_qa_set("../evals/qa_set.jsonl")
llm = get_llm()  # used for generation (recipe_fn's own LLM calls)

# Judging needs its own backend: under a real API key this is the same
# real model, but under mock, `llm`'s canned generation text isn't valid
# JSON, and the judge prompts require JSON output. A separate MockLLM
# here demonstrates a clean, illustrative run instead of every question
# correctly (but noisily) failing to parse -- see evals/judges.py's
# JudgeParseError and evals/run.py's per-question error isolation.
if os.environ.get("RAG_RECIPES_LLM", "openai").lower() == "mock":
    judge_llm = MockLLM(default_response='{"score": 1, "reasoning": "Mock judge: looks fine."}')
else:
    judge_llm = llm

from recipes.embeddings import get_embedder

embedder = get_embedder()
trace_output_path = "../outputs/agentic_traces.jsonl"


## 1. What this pattern does

An LLM-driven loop (`prompts/agentic_prompt.txt`) decides, turn by turn, which tool to call --
`search_dense`, `search_bm25`, or `finish` -- rather than following a fixed retrieval strategy. The
agent's job is retrieval orchestration ONLY: `finish` ends the search loop but does not produce the
answer itself, so the final answer always goes through the exact same R4 held-constant generation
prompt every other pattern uses (see `recipes/agentic.py`'s module docstring for why -- keeping R4
airtight with zero exceptions was a deliberate design decision, not an oversight). Every step
(thought, action, action_input, observation) is recorded into `tool_call_trace`.


## 2. When to use it

- You don't know in advance what retrieval strategy (keyword vs. semantic, how many searches) a
  given question needs -- letting the model decide adaptively can outperform any single fixed
  pattern across a diverse question mix
- You want visibility into the retrieval reasoning itself (the trace), not just the final answer
- You can afford a variable, potentially larger number of LLM calls per question (bounded by
  `MAX_ITERATIONS`, default 4)


## 3. When NOT to use it

- SPEC.md's key claim for this pattern is blunt: "Most flexible. Hardest to debug." A malformed or
  looping agent is genuinely hard to diagnose compared to any fixed-strategy pattern -- `recipes/agentic.py`'s
  `_parse_agent_action()` defensively forces a `finish` on any unparseable response specifically to
  bound this risk, but the underlying unpredictability is real
- Cost/latency budget requires a predictable, fixed number of retrieval calls per question
- Your question mix is uniform enough that a fixed pattern (e.g. hybrid+rerank) already performs
  well -- the agent's adaptivity is wasted when there's nothing to adapt to


## 4. Implementation

In [3]:
from recipes.agentic import make_retrieve_and_answer

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)

# Try it on one question directly.
sample = retrieve_and_answer("What does PCEval stand for?", k=3)
print("retrieved:", sample.retrieved_chunk_ids)
print("answer:", sample.answer)
print("tool_call_trace steps:", len(sample.tool_call_trace))


retrieved: []
answer: This is a mock response.
tool_call_trace steps: 1


## 5. Run on our eval set

In [4]:
pattern_fn = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)

result = run_pattern(
    recipe_fn=pattern_fn,
    qa_set=qa_set,
    corpus_by_id=corpus_by_id,
    llm=judge_llm,
    pattern_name="10_agentic",
    judges_enabled=True,
    trace_output_path=trace_output_path,
)

import os
print()
print(f"wrote traces to {trace_output_path}: {os.path.exists(trace_output_path)}")
with open(trace_output_path, encoding="utf-8") as f:
    n_trace_lines = sum(1 for _ in f)
print(f"trace lines written: {n_trace_lines}")


=== 10_agentic (n=18) ===
  hit@3: 0.000  [95% CI 0.000, 0.000]
  hit@10: 0.000  [95% CI 0.000, 0.000]
  mrr: 0.000  [95% CI 0.000, 0.000]
  faithfulness: 1.000  [95% CI 1.000, 1.000]
  answer_relevance: 1.000  [95% CI 1.000, 1.000]
  citation_accuracy: 1.000  [95% CI 1.000, 1.000]
  filter_accuracy: 0.000  [95% CI 0.000, 0.000]
  p50_latency_ms: 0.1
  p95_latency_ms: 0.2
  usd_per_query: $0.00014
  eval_usd: $0.0026

wrote traces to ../outputs/agentic_traces.jsonl: True
trace lines written: 18


## 6. Example query walkthrough

One example per eval-set category, showing the retrieved chunks, the (mocked) final answer, AND the
full agent trace -- seeing the agent's reasoning is the whole point of this pattern.

In [5]:
examples = {
    "keyword": "What does PCEval stand for?",
    "paraphrase": "Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?",
    "multi_hop": "The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?",
    "filter": "Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?",
}

for category, question in examples.items():
    result = retrieve_and_answer(question, k=3)
    print(f"--- {category} ---")
    print(f"Q: {question}")
    print(f"Retrieved: {result.retrieved_chunk_ids}")
    print(f"A: {result.answer}")
    print("Trace:")
    for step in result.tool_call_trace:
        print(f"  {step}")
    print()


--- keyword ---
Q: What does PCEval stand for?
Retrieved: []
A: This is a mock response.
Trace:
  {'thought': '(unparseable response, stopping)', 'action': 'finish', 'action_input': '', 'observation': None}

--- paraphrase ---
Q: Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?
Retrieved: []
A: This is a mock response.
Trace:
  {'thought': '(unparseable response, stopping)', 'action': 'finish', 'action_input': '', 'observation': None}

--- multi_hop ---
Q: The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?
Retrieved: []
A: This is a mock response.
Trace:
  {'thought': '(unparseable response, stopping)', 'action': 'finish', 'action_input': '', 'observation': None}

--- filter ---
Q: Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?
Retrieved: []
A: This is a mock respons

## 7. Where this pattern FAILS

**PENDING: real findings from a one-off real-embeddings run.** A real `OPENAI_API_KEY` was not yet
available in the environment when this notebook was authored. This section will be replaced with a
static table of genuine hit@k/mrr failures (matching the format used in `02_bm25.ipynb`/
`04_rerank.ipynb` section 7), computed via a real, uncommitted exploratory run once a key is
available -- see `tasks/todo.md` for tracking. The claim will be labeled with the date it was run
and its actual dollar cost, and will not be presented as live-executed cell output, to avoid
implying a mock re-run reproduces it (see this notebook's top-of-file disclaimer).


## 8. Copy-paste snippet

Meant for pasting into your own project, not executed as a cell in this notebook.

```python
"""Minimal agentic retrieval + generation, no eval harness."""
from recipes.embeddings import get_embedder
from recipes.agentic import make_retrieve_and_answer
from recipes.llm import get_llm

corpus_by_id = {}  # {chunk_id: {"text": ..., ...}, ...} -- fill in your own chunks
embedder = get_embedder()
llm = get_llm()

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)
result = retrieve_and_answer("your question here", k=5)
print(result.answer)
print(result.tool_call_trace)
```
